# Notebook 02 — CP01: Data Generation and QC

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 2 of 12  
**Time estimate:** 60 minutes

> CP01 simulates an end-to-end RNA-seq differential expression study.
> We use synthetic negative-binomial count data because raw FASTQ processing
> requires HPC resources beyond this programme. The statistical analysis is
> identical to what you would run on real data.

---
## Step 1 — Biological Context

**Question:** Which genes are differentially expressed between control and treated cells?

**Design:** 3 control samples vs 3 treated samples (e.g., DMSO vs drug X at 24h).
Each sample is a count matrix: rows = genes, columns = samples.

**Why negative-binomial?** RNA-seq counts are overdispersed relative to Poisson
(i.e., variance > mean). The negative-binomial distribution captures this.
DESeq2 (Love et al. 2014) models counts as NB with gene-specific dispersion.

**NB parameterization used here:**
- Mean expression per gene: `μ ~ LogNormal(3, 1.5)` (spans low to highly expressed)
- Dispersion per gene: `α ~ Exponential(0.1) + 0.01` (small = tight, large = noisy)
- Counts: `NB(r = 1/α, p = 1/(1 + α*μ))` where r = size, p = success probability
- 300 DE genes injected: 150 upregulated (2–4× fold), 150 downregulated (÷2 to ÷4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib.patches as mpatches

rng = np.random.default_rng(42)

# ---- Generate synthetic RNA-seq count matrix ----

N_GENES   = 5000
N_SAMPLES = 6  # 3 control (cols 0-2) + 3 treated (cols 3-5)
N_DE      = 300

# Gene-level parameters
mu_base    = rng.lognormal(mean=3.0, sigma=1.5, size=N_GENES)   # mean expression
dispersion = rng.exponential(scale=0.1, size=N_GENES) + 0.01    # NB dispersion

# Counts: NB(r=1/alpha, p=1/(1+alpha*mu))
r = 1.0 / dispersion[:, None]  # (n_genes, 1)
p = 1.0 / (1.0 + dispersion[:, None] * mu_base[:, None])
counts = rng.negative_binomial(r, p, size=(N_GENES, N_SAMPLES)).astype(float)

# Inject DE genes: 150 up, 150 down in treated samples (cols 3-5)
de_idx = rng.choice(N_GENES, N_DE, replace=False)
up_idx, dn_idx = de_idx[:N_DE//2], de_idx[N_DE//2:]
fold_changes = rng.uniform(1.5, 4.0, N_DE)
counts[up_idx, 3:] = (counts[up_idx, 3:] * fold_changes[:N_DE//2, None]).astype(int)
counts[dn_idx, 3:] = np.maximum(1, (counts[dn_idx, 3:] / fold_changes[N_DE//2:, None]).astype(int))
counts = np.clip(counts, 0, None)

# Build DataFrame
gene_names  = [f'GENE_{i:04d}' for i in range(N_GENES)]
sample_names = [f'ctrl_{i+1}' for i in range(3)] + [f'trt_{i+1}' for i in range(3)]
conditions   = ['control']*3 + ['treated']*3

counts_df = pd.DataFrame(counts.astype(int), index=gene_names, columns=sample_names)

print(f'Count matrix: {counts_df.shape[0]} genes × {counts_df.shape[1]} samples')
print(f'Library sizes: {counts_df.sum(axis=0).values}')
print(f'DE genes injected: {N_DE} ({N_DE//2} up, {N_DE//2} down in treated)')
print(f'Median counts per gene: {counts_df.median(axis=1).median():.1f}')

In [ ]:
# ---- QC metrics ----

# Detected genes (count > 0)
detected = (counts_df > 0).sum(axis=0)

# Log-normalize (log2 CPM)
lib_sizes  = counts_df.sum(axis=0)
log2_cpm   = np.log2(counts_df.divide(lib_sizes) * 1e6 + 1)

# PCA (manual, 2 PCs)
X = log2_cpm.T.values  # (6, 5000)
X_centered = X - X.mean(axis=0)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
pc1, pc2 = X_centered @ Vt[0], X_centered @ Vt[1]
var_explained = (S**2 / (S**2).sum())[:2] * 100

# Pearson correlation matrix
corr = np.corrcoef(log2_cpm.values.T)  # (6, 6)

print(f'Detected genes per sample (count > 0): {detected.values}')
print(f'PC1 variance: {var_explained[0]:.1f}%, PC2: {var_explained[1]:.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors_cond = ['#4e79a7']*3 + ['#e15759']*3

# Panel A: Library size distribution
ax = axes[0, 0]
ax.bar(range(N_SAMPLES), lib_sizes.values / 1e6, color=colors_cond, edgecolor='black', alpha=0.8)
ax.set_xticks(range(N_SAMPLES)); ax.set_xticklabels(sample_names, fontsize=9)
ax.set_ylabel('Library size (millions)')
ax.set_title('A. Library sizes\n(should be similar across samples)')
for i, ls in enumerate(lib_sizes.values):
    ax.text(i, ls/1e6 + 0.02, f'{ls/1e6:.1f}M', ha='center', fontsize=8)
patch_ctrl = mpatches.Patch(color='#4e79a7', label='Control')
patch_trt  = mpatches.Patch(color='#e15759', label='Treated')
ax.legend(handles=[patch_ctrl, patch_trt], fontsize=9)

# Panel B: Detected genes per sample
ax = axes[0, 1]
ax.bar(range(N_SAMPLES), detected.values, color=colors_cond, edgecolor='black', alpha=0.8)
ax.set_xticks(range(N_SAMPLES)); ax.set_xticklabels(sample_names, fontsize=9)
ax.set_ylabel('Number of detected genes (count > 0)')
ax.set_title('B. Detected genes per sample')
ax.set_ylim(0, N_GENES * 1.1)

# Panel C: PCA
ax = axes[1, 0]
for i, (x, y, cond, sname) in enumerate(zip(pc1, pc2, conditions, sample_names)):
    color = '#4e79a7' if cond == 'control' else '#e15759'
    ax.scatter(x, y, color=color, s=120, zorder=3, edgecolors='black')
    ax.text(x + 0.3, y + 0.3, sname, fontsize=8)
ax.set_xlabel(f'PC1 ({var_explained[0]:.1f}% variance)')
ax.set_ylabel(f'PC2 ({var_explained[1]:.1f}% variance)')
ax.set_title('C. PCA of log2 CPM\n(PC1 separates treatment groups)')
ax.axhline(0, color='grey', ls=':', lw=0.8)
ax.axvline(0, color='grey', ls=':', lw=0.8)
ax.legend(handles=[patch_ctrl, patch_trt], fontsize=9)

# Panel D: Pearson correlation heatmap
ax = axes[1, 1]
im = ax.imshow(corr, cmap='RdBu_r', vmin=0.7, vmax=1.0)
plt.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')
ax.set_xticks(range(N_SAMPLES)); ax.set_xticklabels(sample_names, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(N_SAMPLES)); ax.set_yticklabels(sample_names, fontsize=9)
for i in range(N_SAMPLES):
    for j in range(N_SAMPLES):
        ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center', fontsize=7,
                  color='white' if corr[i,j] < 0.87 else 'black')
ax.set_title('D. Sample correlation heatmap\n(within-group r should be > between-group)')

plt.suptitle('Module 20 CP01: RNA-seq Count Matrix QC', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cp01_qc.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[What does PC1 separation in panel C tell you about the data? What would
> you do if two samples from the same condition clustered far from each other?
> What is the minimum acceptable within-group Pearson r for a real RNA-seq experiment?]*

---
*Next: `03_cp01_differential_expression.ipynb`*